# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RawanMohamed16/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Clustering.** My lane's question is "what kinds of content exist" — that's a
segmentation question, not "will X happen" (classification) or "which one first"
(ranking). There's no observed outcome to predict; the goal is to find groups that
already exist in the data but aren't labeled, so a content strategist can treat
each group differently instead of applying one rule to all 30,000 pages.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# quick check that backs the "clustering, not classification" call:
# there is no ground-truth group/category label in the columns — the closest
# thing (content_type) is only 3 values and doesn't capture behavior.
print(df.shape)
print([c for c in df.columns if "cluster" in c.lower() or "group" in c.lower() or "segment" in c.lower()])
print(df["content_type"].value_counts())


(30000, 44)
[]
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**No target — this is unsupervised.** There's nothing to predict, so instead of
a label I pick a **feature vector** that stands in for "content behavior":
`word_count`, `engagement_rate`, `ctr`, `scroll_rate`, `ai_traffic_pct`,
`avg_position` (numeric, all real 90-day measurements — not defined rules), plus
`content_type` and `main_intent` (categorical). I deliberately leave out
`trend_direction` / `trend_pct` — those are the label for the *classification*
lane (declining vs not) and would leak that framing in here.

Two columns have real missingness that I have to handle honestly rather than
blindly filling: `word_count` is blank for ~7,700 rows and `main_intent` for
~2,400 — both blank in a pattern tied to `content_type`, per the data
dictionary, so a naive `fillna(0)` would quietly encode content type into the
clustering. I use the column median for numerics and an explicit `"unknown"`
category rather than 0/blank.

In [2]:
numeric_feats = ["word_count", "engagement_rate", "ctr", "scroll_rate",
                 "ai_traffic_pct", "avg_position"]
categorical_feats = ["content_type", "main_intent"]

print("missing values in candidate features:")
print(df[numeric_feats + categorical_feats].isna().sum())

# confirm the missingness in word_count lines up with content_type, as the
# data dictionary warns -- this is why I impute per-column instead of a blind fillna(0)
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).round(2))


missing values in candidate features:
word_count         7699
engagement_rate       0
ctr                   0
scroll_rate         125
ai_traffic_pct        0
avg_position          0
content_type          0
main_intent        2374
dtype: int64
content_type
comparison article    0.00
feedly article        0.00
keyword article       0.28
Name: word_count, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Silhouette score, checked alongside a human sense-check of the cluster
profiles.** Silhouette tells me whether the groups are actually separated in
feature space (closer to 1 is better, 0 means overlapping, and it needs no
label to compute — right for unsupervised work). But a high silhouette
score with clusters nobody can name is useless, so I also report each
cluster's size and its average feature values, and check whether a content
strategist could describe each one in a sentence.

Below, k=3 gives the best silhouette (~0.35) among k=3..6 tried, and its three
clusters are actually describable (see section 5), so I pick k=3.

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sub = df[numeric_feats + categorical_feats].copy()
for c in numeric_feats:
    sub[c] = sub[c].fillna(sub[c].median())
for c in categorical_feats:
    sub[c] = sub[c].fillna("unknown")

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_feats),
])
X = pre.fit_transform(sub)

rng = np.random.RandomState(0)
sample_idx = rng.choice(X.shape[0], size=4000, replace=False)

for k in [3, 4, 5, 6]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    sil = silhouette_score(X[sample_idx], labels[sample_idx])
    print(f"k={k}  silhouette~{sil:.3f}  sizes={np.bincount(labels)}")


k=3  silhouette~0.351  sizes=[ 4827 25013   160]


k=4  silhouette~0.336  sizes=[ 4783 24964   159    94]


k=5  silhouette~0.287  sizes=[19369  3257    95  7120   159]


k=6  silhouette~0.290  sizes=[ 7045 18671   131    95  3573   485]


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), aggregated over its trailing
90-day window. That's the grain the clustering runs at — I'm grouping pages,
not clients or days. `client_id` stays out of the feature vector; it's only
for grouping/joins, per the data dictionary.

In [4]:
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns — one row per content_id")
df[["content_id", "client_id", "content_type", "main_intent",
    "word_count", "engagement_rate", "ctr", "ai_traffic_pct"]].head()


30,000 rows, 44 columns — one row per content_id


,content_id,client_id,content_type,main_intent,word_count,engagement_rate,ctr,ai_traffic_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,5.88,0.76,0.0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,0.00,0.05,0.0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,0.00,0.09,0.0
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,1.28,0.49,0.0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,0.00,0.13,0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would need to be something like "if engagement_rate > X and
word_count > Y and ai_traffic_pct > Z, call it group A" — but these signals
don't split cleanly on any single threshold, and they interact: below, the
three clusters KMeans finds differ on *combinations* of engagement, traffic
mix, and content_type, not any one column alone. Cluster 2, for instance, is
small (160 pages) and mostly has *unknown* `main_intent` — a rule-writer would
have had to notice that missing-metadata pattern and hand-code it, which is
exactly the kind of messy, many-signal, not-obvious-in-advance structure
clustering is built to surface instead of guessing at by hand.

In [5]:
best_k = 3
labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X)
sub["cluster"] = labels

print(sub["cluster"].value_counts(), "\n")
print("mean feature values per cluster:")
print(sub.groupby("cluster")[numeric_feats].mean().round(2), "\n")
print("content_type mix per cluster:")
print(sub.groupby("cluster")["content_type"].value_counts(normalize=True).round(2))


cluster
1    25013
0     4827
2      160
Name: count, dtype: int64 

mean feature values per cluster:
         word_count  engagement_rate    ctr  scroll_rate  ai_traffic_pct  \
cluster                                                                    
0           2578.45             7.33   0.39        73.05            1.95   
1           3150.23             1.56   0.29         7.52            0.54   
2           1333.61             9.89  37.94        25.22            1.34   

         avg_position  
cluster                
0               15.57  
1               16.56  
2                6.49   

content_type mix per cluster:
cluster  content_type      
0        keyword article       0.73
         feedly article        0.17
         comparison article    0.09
1        keyword article       0.94
         feedly article        0.05
         comparison article    0.01
2        feedly article        0.63
         keyword article       0.37
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.